## BitPack Real Machine Evaluation
In this notebook, the `eval.json` configuration file is read and multiple rounds of evaluation are performed according to the configuration.


In [ ]:
# Load the hardware overlay
from pynq import Overlay
import bitpack_lib

pl = Overlay("bitpack_sample.bit")
core = pl.bitpack_top_0

In [ ]:
# Read the configuration file
import bitpack_eval_config as bpconf
import bitpack_lib_io as bpio
import itertools

conf = bpconf.BitPackEvalConfig("eval.json")
ivalues = [[bpio.BitPackInput.RANDOM] if d.random else d.values for d in conf.inputs]
combos = list(itertools.product(*ivalues))

In [ ]:
# Set up RNG
import bitpack_lib_rand as bprand

rng = bprand.BitPackRandom(conf.seed)
core.setrng(rng)

In [ ]:
# Evaluate
f_result = open('results.txt', 'w')
core.cycle = conf.cycle
for i in range(conf.times):
    for combo in combos:
        core.srcs.values = combo
        core.resetseeds()
        core.start()
        v_result = [p.getvaluereal() for p in core.srcs] + [p.value for p in core.dsts]
        print(' '.join([f"{v:.6g}" for v in v_result]), file=f_result)
        
f_result.close()
print(f"## Circuit completes {len(combos)} x {conf.times} times of evaluation")